# BFS & DFS Pattern Playbook - Trees
**Topic:** Trees | **Type:** Pattern notebook (own one baseline, get several problems as small edits)

This notebook does for BFS/DFS "early-stop" and "path" problems exactly what section **2.2 -> 2.3** of
`0. tree_traversals_dfs_and_bfs.ipynb` did for level-order BFS: write one baseline function in full, then
show that a cluster of real LeetCode problems are each just a **small, callable-out edit** to that same
function - no extra machinery, no generic engine to learn. Every function below is complete and
self-contained, so you can read any one of them top to bottom without needing to jump elsewhere first.

- **Part 1 - BFS:** one baseline (Minimum Depth), two variants that extend it (All Nodes Distance K,
  and a stack-safety demonstration on a skewed tree).
- **Part 2 - DFS, "compute a value from children":** one baseline (Maximum Depth), three variants
  (Balanced Tree, Diameter, Path Sum).
- **Part 3 - DFS, "backtrack along root-to-leaf paths":** one baseline (Binary Tree Paths), three variants
  (Path Sum II, Sum Root to Leaf Numbers, Smallest String From Leaf).
- **Part 4 - DFS, "sorted order on a BST":** one baseline (Validate BST), one variant (Kth Smallest in a BST).


In [2]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val; self.left = left; self.right = right

def build_tree(values):
    """Level-order list -> tree, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values):
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def find_node(root, val):
    """Test helper only - LeetCode hands you the node directly, not a value to search for."""
    if not root: return None
    if root.val == val: return root
    return find_node(root.left, val) or find_node(root.right, val)


---
# Part 1 - BFS

## 1.1 Minimum Depth of Binary Tree (LC 111) - the baseline

**Idea:** BFS explores ring by ring, so the *first* leaf it dequeues is guaranteed to be the **nearest**
one - no need to check every path and keep a running minimum, the way DFS would have to. This is the
worked example from notebook `0.`, sections 2.4/2.5, written out directly.

**Time:** `O(n)` worst case. **Space:** `O(w)`.


In [3]:
def min_depth(root):
    """BFS: the FIRST leaf dequeued is the NEAREST leaf, guaranteed."""
    if not root:
        return 0
    q = deque([(root, 1)])                 # (node, depth) - root counts as depth 1
    while q:
        node, depth = q.popleft()
        if not node.left and not node.right:
            return depth                    # first leaf popped -> nearest leaf
        if node.left:
            q.append((node.left, depth + 1))
        if node.right:
            q.append((node.right, depth + 1))


t = build_tree([1, 2, 3, 4, 5, None, None, None, None, None, 6])   # the tree from notebook 0's example
print("min depth:", min_depth(t))                   # expect 2 (leaf `3` sits one edge from the root)
print("single node:", min_depth(build_tree([1])))   # expect 1
print("empty tree:", min_depth(None))               # expect 0


min depth: 2
single node: 1
empty tree: 0


## 1.2 All Nodes Distance K in Binary Tree (LC 863)

**Same as baseline, except:**
1. BFS starts from `target`, not `root`.
2. It can move in **three** directions instead of two - `left`, `right`, **and `parent`** (a `TreeNode`
   only gives you the first two for free, so one quick pre-pass builds a `child -> parent` map).
3. There's no early stop - we want **every** node at distance `k`, not just the first match, so instead of
   `return`-ing the moment we find one, we collect it and keep going.

Nothing about the *queue loop itself* changes - it's still "pop, check, push unvisited neighbours." The
only real news is that a tree can be walked **upward** too, once you hand it the one edge it doesn't give
you by default.

**Time:** `O(n)` (the parent pre-pass, plus one BFS). **Space:** `O(n)`.


In [ ]:
def build_parent_map(root):
    """One traversal: map every child node to its parent (the root has no entry)."""
    parent = {}
    stack = [root]
    while stack:
        node = stack.pop()
        if node.left:
            parent[node.left] = node; 
            stack.append(node.left)
        if node.right:
            parent[node.right] = node; 
            stack.append(node.right)
    return parent


def distance_k(root, target, k):
    """BFS from `target`, moving left/right/parent, collecting every node at distance k."""
    parent = build_parent_map(root)         # CHANGE 1: need parent pointers to move "upward"
    visited = {target}
    q = deque([(target, 0)])                # CHANGE 2: start from `target`, not `root`
    result = []
    while q:
        node, dist = q.popleft()
        if dist == k:
            result.append(node.val)          # CHANGE 3: collect it instead of returning immediately
            continue                          # a match has nothing useful left to explore
        for neighbor in (node.left, node.right, parent.get(node)):   # CHANGE 1, continued: 3 directions
            if neighbor and neighbor not in visited:
                visited.add(neighbor)
                q.append((neighbor, dist + 1))
    return result


tk = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
target = find_node(tk, 5)                   # LeetCode hands you the node directly; we search for it here
print(sorted(distance_k(tk, target, 2)))    # expect [1, 4, 7]


[1, 4, 7]


In [6]:
tk

## 1.3 Bonus: BFS never recurses, so it never blows the call stack

**Same as baseline, except:** nothing about the code changes at all - this section is here to make a claim
from the decision table concrete, not to introduce a new function. `min_depth`'s queue lives on the **heap**
(an ordinary Python list inside a `deque`), not on the **call stack**. A recursive traversal, by contrast,
adds one stack frame per level of depth - so on a tall, narrow ("skewed") tree, recursion can hit Python's
recursion limit (~1000 frames) while the exact same walk done with an explicit queue is untroubled.

This is why "deep/skewed tree, might blow the call stack" is its own row in the decision table: it isn't
about *which answer* BFS vs. DFS gives - both give the same answer here - it's about which one *survives*
being asked the question in the first place.


In [5]:
def left_chain(n):
    """1 -> 2 -> ... -> n, all LEFT children - a maximally skewed tree, height n."""
    root = TreeNode(1); cur = root
    for v in range(2, n + 1):
        cur.left = TreeNode(v); cur = cur.left
    return root


def min_depth_recursive(root):
    """The natural RECURSIVE version of min_depth - included ONLY to show it can blow the stack."""
    if not root:
        return 0
    if not root.left and not root.right:
        return 1
    return 1 + min(min_depth_recursive(c) for c in (root.left, root.right) if c)


deep = left_chain(5000)                    # height 5000 -> 5000 nested calls if done recursively

try:
    print("recursive min depth:", min_depth_recursive(deep))
except RecursionError:
    print("recursive min depth: RecursionError - blew the call stack")

print("BFS min depth:       ", min_depth(deep))     # same function as 1.1 - completely unaffected


recursive min depth: RecursionError - blew the call stack
BFS min depth:        5000


## 1.4 Recap - all four BFS signals from the decision table

| Decision-table signal | Demonstrated in |
|---|---|
| "shortest path / minimum depth / fewest hops" | **1.1** `min_depth` |
| "level by level", "nearest to the root" | notebook `0.`, sections 2.2-2.3 (`bfs_levels` and its variants) - not repeated here |
| a step needs a direction `.left`/`.right` doesn't give you | **1.2** `distance_k` |
| deep/skewed tree, avoid blowing the call stack | **1.3** `min_depth` vs. `min_depth_recursive` |

Same queue-loop skeleton every time; only *where it starts*, *which edges it's allowed to use*, *whether it
stops early*, and *why you'd bother using a queue at all* changed.


---
# Part 2 - DFS: compute a value from children

## 2.1 Maximum Depth of Binary Tree (LC 104) - the baseline

**Idea:** handle the base case, recurse into both children, **combine** their answers into this node's
answer. The shortest possible DFS shape.

**Time:** `O(n)`. **Space:** `O(h)`.


In [ ]:
def max_depth(node):
    if not node:
        return 0
    return 1 + max(max_depth(node.left), max_depth(node.right))


t104 = build_tree([3, 9, 20, None, None, 15, 7])
print(max_depth(t104))             # expect 3
print(max_depth(None))             # expect 0


## 2.2 Balanced Binary Tree (LC 110)

**Same as baseline, except:** the return value is **overloaded** with a sentinel - `-1` means "already
found an imbalance below, stop bothering to compute further." That one sentinel turns "compute height at
every node separately" (`O(n^2)`) into a single `O(n)` pass, because a `-1` short-circuits every ancestor's
check on the way back up.


In [ ]:
def is_balanced(root):
    def height(node):
        if not node:
            return 0
        lh = height(node.left)
        if lh == -1:
            return -1                          # left subtree already unbalanced - bail out early
        rh = height(node.right)
        if rh == -1:
            return -1
        if abs(lh - rh) > 1:
            return -1                          # THIS node is the one that's unbalanced
        return 1 + max(lh, rh)                 # otherwise identical to max_depth
    return height(root) != -1


print(is_balanced(build_tree([3, 9, 20, None, None, 15, 7])))        # expect True
print(is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4])))    # expect False


## 2.3 Diameter of Binary Tree (LC 543)

**Same as baseline, except:** the function still computes height exactly like `max_depth` - but along the
way, it stashes the **best `left_height + right_height` seen at any node** into a `nonlocal` variable. The
diameter (longest path between any two nodes) doesn't have to pass through the root, so it can't be the
*return value* - it has to be tracked as a side effect while the height recursion runs anyway.


In [ ]:
def diameter_of_binary_tree(root):
    best = 0
    def height(node):
        nonlocal best
        if not node:
            return 0
        lh = height(node.left)
        rh = height(node.right)
        best = max(best, lh + rh)              # longest path THROUGH this node, checked at every node
        return 1 + max(lh, rh)                 # still just max_depth's return value
    height(root)
    return best


print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))   # expect 3


## 2.4 Path Sum (LC 112)

**Same as baseline, except:** instead of *returning* information up from children, we *pass* information
**down** to them (the remaining sum needed) - and the base case fires only at a true leaf, not at `None`.
Still "handle base case, recurse into children, combine" - just top-down instead of bottom-up.


In [ ]:
def has_path_sum(root, target):
    if not root:
        return False
    if not root.left and not root.right:       # base case: a LEAF, not an empty subtree
        return root.val == target
    remaining = target - root.val               # pass state DOWN instead of combining state UP
    return has_path_sum(root.left, remaining) or has_path_sum(root.right, remaining)


t112 = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print(has_path_sum(t112, 22))                 # expect True  (5 -> 4 -> 11 -> 2)
print(has_path_sum(build_tree([1, 2]), 1))    # expect False


## 2.5 Recap

| Problem | Same as baseline, except... |
|---|---|
| 104 Max Depth | *(this IS the baseline)* |
| 110 Balanced Tree | overload the return value with a `-1` sentinel to short-circuit |
| 543 Diameter | track a `nonlocal` best-so-far alongside the same height recursion |
| 112 Path Sum | pass state down (remaining sum) instead of combining state up |


---
# Part 3 - DFS: backtrack along root-to-leaf paths

## 3.1 Binary Tree Paths (LC 257) - the baseline

**Idea:** **choose** a child, **recurse** into it with that child appended to the running path, then
**un-choose** it (pop) once that branch is fully explored. Record the finished path only when a **leaf** is
reached.


In [ ]:
def binary_tree_paths(root):
    if not root:
        return []
    out, path = [], [str(root.val)]
    def backtrack(node):
        if not node.left and not node.right:
            out.append("->".join(path))         # RECORD - reached a leaf
            return
        for nxt in (node.left, node.right):
            if nxt:
                path.append(str(nxt.val))         # CHOOSE
                backtrack(nxt)                      # RECURSE
                path.pop()                          # UN-CHOOSE
    backtrack(root)
    return out


t257 = build_tree([1, 2, 3, None, 5])
print(binary_tree_paths(t257))     # expect ['1->2->5', '1->3']


## 3.2 Path Sum II (LC 113)

**Same as baseline, except:** the path stores raw `int`s instead of strings (no need to join them at the
end), and we track a `remaining` sum alongside it. The "record" condition adds one clause - it's a leaf
**and** `remaining == 0` - otherwise the un-choose (`path.pop()`) still happens exactly like the baseline.


In [ ]:
def path_sum_ii(root, target):
    out, path = [], []
    def backtrack(node, remaining):
        if not node:
            return
        path.append(node.val)                    # CHOOSE
        remaining -= node.val
        if not node.left and not node.right and remaining == 0:
            out.append(path[:])                    # RECORD - leaf AND target hit exactly
        else:
            backtrack(node.left, remaining)          # RECURSE
            backtrack(node.right, remaining)
        path.pop()                                # UN-CHOOSE (always, whether or not we recorded)
    backtrack(root, target)
    return out


t113 = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
print(path_sum_ii(t113, 22))       # expect [[5, 4, 11, 2], [5, 8, 4, 5]]


## 3.3 Sum Root to Leaf Numbers (LC 129)

**Same as baseline, except:** there's no `path` list at all - the running path is folded into a single
number (`current = current * 10 + node.val`) that lives as a plain function argument, so there's nothing to
explicitly "un-choose": each recursive call gets its own `current`, which simply evaporates when that call
returns. Recording (`total += current`) happens at every leaf, into a `nonlocal` total - the same trick
section 2.3 used for diameter.


In [ ]:
def sum_root_to_leaf(root):
    total = 0
    def backtrack(node, current):
        nonlocal total
        if not node:
            return
        current = current * 10 + node.val         # CHOOSE - folded into the running number
        if not node.left and not node.right:
            total += current                        # RECORD - reached a leaf
            return
        backtrack(node.left, current)                # RECURSE (no explicit pop - `current` just resets per call)
        backtrack(node.right, current)
    backtrack(root, 0)
    return total


print(sum_root_to_leaf(build_tree([1, 2, 3])))     # expect 25  (12 + 13)


## 3.4 Smallest String Starting From Leaf (LC 988)

**Same as baseline, except:** two small twists. The path is still built root-to-leaf, but read out
**reversed** at each leaf (the problem wants leaf-to-root strings). And "record" doesn't just append - it
keeps a running best (`best[0]`), comparing the new candidate lexicographically. Same choose/recurse/
un-choose skeleton as `binary_tree_paths`, with the record step doing a comparison instead of a plain
append.


In [ ]:
def smallest_from_leaf(root):
    best = [None]                                # mutable box - simplest way to track "best so far" here
    path = []
    def backtrack(node):
        if not node:
            return
        path.append(chr(ord("a") + node.val))     # CHOOSE
        if not node.left and not node.right:
            s = "".join(reversed(path))             # RECORD - leaf-to-root, so reverse the root-to-leaf path
            if best[0] is None or s < best[0]:
                best[0] = s
        else:
            backtrack(node.left)                     # RECURSE
            backtrack(node.right)
        path.pop()                                  # UN-CHOOSE
    backtrack(root)
    return best[0]


t988 = build_tree([0, 1, 2, 3, 4, 3, 4])
print(smallest_from_leaf(t988))    # expect 'dba'


## 3.5 Recap

| Problem | Same as baseline, except... |
|---|---|
| 257 Binary Tree Paths | *(this IS the baseline)* |
| 113 Path Sum II | track a numeric `remaining`; record only on leaf AND `remaining == 0` |
| 129 Sum Root to Leaf Numbers | fold the path into one number passed as an argument - no explicit pop needed |
| 988 Smallest String From Leaf | reverse the path at record time; keep a running "best so far" instead of collecting all paths |


---
# Part 4 - DFS: sorted order on a BST

## 4.1 Validate Binary Search Tree (LC 98) - the baseline

**Idea:** a **BST** means "left subtree < node < right subtree," everywhere. Checking that directly at
every node is fiddly (a node deep in the left subtree must be less than *every* ancestor above it, not just
its immediate parent). The clean way in: **inorder traversal visits a BST's values in sorted order** - so
instead of comparing subtree ranges, just walk inorder and check that the value **strictly increases** each
step, using one running `prev`.

**Time:** `O(n)`. **Space:** `O(h)`.


In [ ]:
def is_valid_bst(root):
    """Inorder visits BST values in sorted order - track `prev` and require it to strictly increase."""
    prev = [None]                             # mutable box - "the last value we saw", or None at the start
    def inorder(node):
        if not node:
            return True
        if not inorder(node.left):
            return False                        # LEFT must already be valid (and end below `node.val`)
        if prev[0] is not None and node.val <= prev[0]:
            return False                        # not strictly increasing -> not a BST
        prev[0] = node.val
        return inorder(node.right)              # RIGHT must continue increasing from here
    return inorder(root)


print(is_valid_bst(build_tree([2, 1, 3])))                          # expect True
print(is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])))        # expect False
print(is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7])))        # expect False - 3 is a right-subtree node but < 5


## 4.2 Kth Smallest Element in a BST (LC 230)

**Same as baseline, except:** it's still an inorder walk, but the thing being tracked changes - instead of
comparing each value to a running `prev`, we count how many nodes inorder has visited so far and stop the
instant that count hits `k`. Inorder gives values in ascending order for free, so "the k-th node visited" *is*
"the k-th smallest value" - no sorting required.


In [ ]:
def kth_smallest(root, k):
    """Same inorder walk as is_valid_bst - except we count visits and stop at the k-th, instead of
    comparing to a running previous value."""
    count = [0]
    result = [None]
    def inorder(node):
        if not node or result[0] is not None:
            return                                # already found it - stop exploring
        inorder(node.left)
        if result[0] is not None:
            return
        count[0] += 1                              # CHANGE: count visits instead of comparing to prev
        if count[0] == k:
            result[0] = node.val                    # CHANGE: record and stop, instead of continuing
            return
        inorder(node.right)
    inorder(root)
    return result[0]


bst = build_tree([5, 3, 6, 2, 4, None, None, 1])
print(kth_smallest(bst, 3))        # expect 3


## 4.3 Recap

| Problem | Same as baseline, except... |
|---|---|
| 98 Validate BST | *(this IS the baseline)* |
| 230 Kth Smallest in a BST | count visits instead of comparing to `prev`; stop and record at the k-th visit |

Both walks are inorder DFS with a small piece of state (`prev`, or `count`) riding along in a mutable box -
the same trick section 2.3's `diameter_of_binary_tree` used with `nonlocal`.


## 🧩 Patterns Learned

- **Own one baseline function, in full - not a generic engine.** Every variant above is a complete,
  standalone function you could paste into an interview and read top to bottom. The "pattern" is something
  *you* notice by diffing two real functions against each other, the same way section 2.3 diffed
  `right_side_view` against `bfs_levels`.
- **BFS's early-stop guarantee extends beyond "stop at the first leaf."** Section 1.2 shows the exact same
  queue loop works when the walk needs to move in a direction the data structure doesn't hand you for free
  (parent-ward) - you just widen `neighbors` and hand-build the missing edge. Section 1.3 shows the same
  queue loop is also immune to a problem DFS can actually have (blowing the call stack on a skewed tree) -
  covering all four BFS rows of notebook `0.`'s section 2.5 decision table.
- **DFS that "computes a value," DFS that "backtracks along a path," and DFS that "walks inorder" are three
  different shapes** - the first combines children's return values (or threads state downward), the second
  explicitly builds and un-builds a `path` (or a folded equivalent) as it recurses, and the third relies on
  inorder specifically visiting a BST in sorted order, tracking one small piece of state (`prev`, or a
  count) as it goes. Recognising which shape a question wants is most of the work; the code itself is a
  small, predictable edit once you know which baseline to start from.
- **This is section 2.2 -> 2.3, applied one level up:** own the *concrete baseline*, and the *variant* is a
  one-or-two-line diff you can predict before you write it - and every row of the BFS-vs-DFS decision table
  now has a concrete function behind it in this notebook.
